<a href="https://colab.research.google.com/github/gustavobraga-byte/PesquisAI/blob/main/PesquisAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 PesquisAI — Agente de IA para Pesquisadores Científicos

O **PesquisAI** é um assistente de inteligência artificial executado diretamente no Google Colab, projetado para automatizar o fluxo de trabalho acadêmico de pesquisadores brasileiros — desde a revisão bibliográfica até a preparação para a submissão de artigos.

### 🛡️ Vantagens
* **100% Gratuito:** Executado na infraestrutura em nuvem do Google.
* **Sem Instalação:** Pronto para uso, direto no seu navegador.
* **Foco Nacional:** Integrado com ferramentas de dados brasileiras (**IBGE** e **DataSUS**).
* **Base Tecnológica:** Desenvolvido sob a arquitetura do **OpenCode**.

---

### 🚀 Como Iniciar

1. No menu superior, clique em **Ambiente de execução** ➡️ **Executar tudo** (ou pressione `Ctrl + F9`).
2. Aguarde cerca de **2 minutos** enquanto as dependências e *skills* são configuradas.
3. Role até o final da página e clique nos botões azuis:
    * **"🤖 Abrir o PesquisAI"** para iniciar a interface.
    * **"📁 Abrir a pasta de trabalho no Google Drive"** para acessar sua pasta de arquivos.

In [ ]:
!curl -fsSL https://opencode.ai/install | bash

In [ ]:
# Precisa disso para que o recurso de selecionar e copiar funcione
!sudo apt-get update && sudo apt-get install -y xclip xsel

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!# Instalando a minha Skill do IBGE
!git clone https://github.com/gustavobraga-byte/Skill-IBGE.git ~/.agents/skills/ibge-br

In [ ]:
# Instalando a skill do OpenDataSUS
# Baixe o repositório
!git clone https://github.com/gustavobraga-byte/Skill-DataSus.git /tmp/opendatasus
# Copie para o diretório de skills
!cp -r /tmp/opendatasus ~/.agents/skills/

In [ ]:
# Instalando as skills do Kdense para pesquisa
# Baixe o repositório
!git clone https://github.com/K-Dense-AI/scientific-agent-skills.git /tmp/scientific
# Copie para o diretório de skills
!cp -r /tmp/scientific/scientific-skills ~/.agents/skills/

In [ ]:
# Instalando o Harness
# Baixe o repositório
!git clone https://github.com/gustavobraga-byte/PesquisAI.git /tmp/harness
# Listar o conteúdo para depuração
!ls -l /tmp/harness
# Copie para o diretório de skills
!cp /tmp/harness/AGENTS.md /content

In [ ]:
from google.colab import drive, auth
from IPython.display import HTML, display
from googleapiclient.discovery import build
import os

# Define the target folder in Google Drive and its local mount point
drive_folder_path = 'My Drive/PesquisAI'

# 1. Conecta o Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    # Correctly define caminho_pasta to reflect the actual mounted path for 'My Drive/PesquisAI'
    caminho_pasta = os.path.join('/content/drive', drive_folder_path)
    print(f"Drive montado. Caminho da pasta '{drive_folder_path}' definido como '{caminho_pasta}'.")
except Exception as e:
    print(f"Erro ao montar o Google Drive: {e}.")

# Autentica a API do Drive com a conta atual do Colab
auth.authenticate_user()

# CORREÇÃO: Usar 'v3' em vez de '3'
service = build('drive', 'v3')

# 2. Skip local folder creation with os.makedirs on mounted Drive, as it's not supported.
# The folder will be handled by Google Drive itself.
# If the folder doesn't exist, it will be implicitly created when files are written to it.
# We can still check for existence or list its contents via Drive API if needed.

# 3. Busca o ID web real da pasta no Google Drive
url_direta = "https://drive.google.com"  # Link reserva caso falhe
try:
    # The query for the Drive API is correct, searching by the folder's name
    query = f"name = '{drive_folder_path.split('/')[-1]}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    resultado = service.files().list(q=query, fields="files(id)").execute()
    arquivos_drive = resultado.get('files', [])

    if arquivos_drive:
        # Acessar o primeiro item da lista mapeada
        id_pasta = arquivos_drive[0]['id']
        url_direta = f"https://drive.google.com/drive/folders/{id_pasta}" # Corrected URL format
        print("Link direto para a pasta gerado com sucesso!")
    else:
        print(f"Pasta '{drive_folder_path.split('/')[-1]}' não encontrada no Google Drive. Usando link geral do Meu Drive.")
except Exception as e:
    print(f"Não foi possível mapear o link direto ({e}). Usando link geral do Meu Drive.")

# 4. Renderiza o botão apontando para a URL exata da pasta
botao_html = f'''
<div style="margin: 10px 0;">
    <a href="{url_direta}" target="_blank" style="text-decoration: none;">
        <button style="
            background: linear-gradient(135deg, #007bff, #00d2ff);
            border: none;
            color: white;
            padding: 14px 28px;
            font-size: 16px;
            font-weight: bold;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            border-radius: 50px;
            cursor: pointer;
            box-shadow: 0 4px 15px rgba(0, 123, 255, 0.3);
            transition: all 0.3s ease;
            display: inline-flex;
            align-items: center;
            gap: 10px;
        " onmouseover="this.style.transform='scale(1.03)'; this.style.boxShadow='0 6px 20px rgba(0, 123, 255, 0.5)';"
           onmouseout="this.style.transform='scale(1)'; this.style.boxShadow='0 4px 15px rgba(0, 123, 255, 0.3)';"
           onmousedown="this.style.transform='scale(0.98)';">
            <span> 📁 </span> Abrir a pasta de trabalho no Google Drive
        </button>
    </a>
</div>
'''

display(HTML(botao_html))

In [ ]:
import os
import subprocess
import time
from google.colab import output
from IPython.display import display, HTML

# 0. INSTALAÇÃO OBRIGATÓRIA (Garante que vai funcionar em qualquer máquina nova)
print("📦 Instalando os paranauê que precisa...")
!apt-get update -qq && apt-get install -y -qq ttyd

# 1. Derruba qualquer terminal antigo para liberar a porta 8000
!pkill -f ttyd
time.sleep(0.5)

# 2. Inicia o ttyd injetando a variável que desativa o bloqueio de mouse do OpenCode
print("🚀 Iniciando o terminal para o PesquisAI..")

# Cria uma cópia do ambiente do sistema e adiciona a flag de desativação
env_config = os.environ.copy()
env_config["OPENCODE_EXPERIMENTAL_DISABLE_COPY_ON_SELECT"] = "1"

subprocess.Popen(
    ["ttyd", "-p", "8000", "bash", "-i", "-c", "opencode; exec bash"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env_config  # Aplica o ambiente modificado aqui
)

time.sleep(2) # Aguarda 2 segundos para o terminal subir totalmente


# 3. Captura a URL real do proxy da porta 8000 do Colab
url = output.eval_js('google.colab.kernel.proxyPort(8000)')

# 4. Renderiza o botão estilizado em HTML/CSS
botao_html = f'''
<div style="margin: 10px 0;">
    <a href="{url}" target="_blank" style="text-decoration: none;">
        <button style="
            background: linear-gradient(135deg, #007bff, #00d2ff);
            border: none;
            color: white;
            padding: 14px 28px;
            font-size: 16px;
            font-weight: bold;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            border-radius: 50px;
            cursor: pointer;
            box-shadow: 0 4px 15px rgba(0, 123, 255, 0.3);
            transition: all 0.3s ease;
            display: inline-flex;
            align-items: center;
            gap: 10px;
        " onmouseover="this.style.transform='scale(1.03)'; this.style.boxShadow='0 6px 20px rgba(0, 123, 255, 0.5)';"
           onmouseout="this.style.transform='scale(1)'; this.style.boxShadow='0 4px 15px rgba(0, 123, 255, 0.3)';"
           onmousedown="this.style.transform='scale(0.98)';">
            <span>  🤖 </span> Abrir o PesquisAI
        </button>
    </a>
</div>
'''

# Exibe o botão na tela
display(HTML(botao_html))

## 📬 Contato e Suporte

👨‍💻 Criado por **Gustavo Bastos Braga** (UFV) — [gustavo.braga@ufv.br](mailto:gustavo.braga@ufv.br)  
⭐ Código-fonte e contribuições: [GitHub PesquisAI](https://github.com/gustavobraga-byte/PesquisAI)
